# Vertex Polishing Demo

This notebook shows the new LPAS refinement layer: adaptive sampling finds a strong near-corner primal sample, then soft active-set reconstruction turns that sample into a verified LP vertex. The goal is to compare the raw sampled point, the polished vertex, and the SciPy/HiGHS optimum on the canonical 2D LP.

In [1]:
from __future__ import annotations

import numpy as np
from pprint import pprint

from lpas.experiments.generators import tiny_known_lp
from lpas.solver.adaptive_solver import AdaptiveLPSolver
from lpas.solver.scipy_handoff import compare_adaptive_to_scipy, solve_with_scipy
from lpas.utils.config import SamplerConfig, SolverConfig, VertexPolishingConfig

np.set_printoptions(suppress=True, precision=6)


In [2]:
problem = tiny_known_lp()
config = SolverConfig(
    batch_size=512,
    max_iter=80,
    patience=20,
    sampler=SamplerConfig(seed=42, sigma_init=2.5, primal_init_mean=0.75, dual_init_mean=0.75),
    vertex_polishing=VertexPolishingConfig(elite_fraction=0.2, max_candidates_per_sample=64, max_total_candidates=256),
)

adaptive = AdaptiveLPSolver(config).solve(problem)
scipy = solve_with_scipy(problem)
_ = compare_adaptive_to_scipy(adaptive, scipy)

pprint(
    {
        'status': adaptive.status.value,
        'solution_source': adaptive.solution_source,
        'raw_best_x': None if adaptive.raw_best_x is None else adaptive.raw_best_x.tolist(),
        'raw_best_objective': adaptive.raw_best_primal_objective,
        'polished_best_x': None if adaptive.polished_best_x is None else adaptive.polished_best_x.tolist(),
        'polished_best_objective': adaptive.polished_best_primal_objective,
        'polished_certified_feasible': adaptive.polished_certified_feasible,
        'raw_vs_scipy_active_set_jaccard': adaptive.raw_vs_scipy_active_set_jaccard,
        'polished_vs_scipy_active_set_jaccard': adaptive.polished_vs_scipy_active_set_jaccard,
        'scipy_x': None if scipy.x is None else scipy.x.tolist(),
        'scipy_objective': scipy.objective,
    }
)


{'polished_best_objective': 10.0,
 'polished_best_x': [2.0, 2.0],
 'polished_certified_feasible': True,
 'polished_vs_scipy_active_set_jaccard': 1.0,
 'raw_best_objective': 9.990371235537367,
 'raw_best_x': [1.9947685874743182, 2.003032736557206],
 'raw_vs_scipy_active_set_jaccard': 0.0,
 'scipy_objective': 10.0,
 'scipy_x': [2.0, 2.0],
 'solution_source': 'vertex_polishing',
 'status': 'APPROXIMATE'}


## Reading The Output

- `raw_best_x` is the best primal-feasible sampled point from the adaptive loop.
- `polished_best_x` is the reconstructed candidate vertex from the soft active-set pass.
- `polished_vs_scipy_active_set_jaccard` shows whether polishing recovered the same active original constraints as HiGHS.
- `solution_source` tells you whether the final reported primal point came from raw sampling or from deterministic vertex polishing.